In [1]:
import os
import cv2
from natsort import natsorted
import glob
import ast

def list_files_with_extension(dir_path, extension):
    '''
    列出指定目录下具有特定后缀的文件。
    
    :param dir_path: 要搜索的目录路径
    :param extension: 文件的后缀名（例如 '.txt'）
    :return: 一个包含符合条件的文件路径列表
    '''
    # 使用 glob 模块匹配通配符路径
    search_path = os.path.join(dir_path, f'*{extension}')
    files = natsorted(glob.glob(search_path))
    return files

def uniform_sample(lst, n):
    # 计算需要跳过的步长
    step = len(lst) / float(n)
    sampled_list = []
    # 使用步长进行均匀采样
    for i in range(n):
        index = int(i * step)
        sampled_list.append(lst[index])
    return sampled_list

def llm(a,b):
    prompt='Please check if the two words {} and {} are synonyms. If they are, return 1. If not, return 0,Please output the result directly'.format(a,b)

    completion = client.chat.completions.create(
        model="GLM-4-AirX",
        messages=[{"role": "user","content": [
            {"type": "text","text": prompt},
        ]}]
    )
    return int(completion.choices[0].message.content)

def acc_(output,label):
    right_1=0
    right_2=0
    right_3=0

    for idx,(i,j) in enumerate(zip(output,label)):
        if idx<10:
            if (i in [True,'yes'] and j in [True,'yes']) or (i in [False,'no'] and j in [False,'no']):
                right_1+=1
        
        if idx>=10 and idx<20:
            if int(i)==int(j):
                right_2+=1

        if idx>=20:
             if llm(i,j)==1:
                right_3+=1

    return right_1,right_2,right_3

In [2]:
path_list=['/data/coding/eval_replica/Replica/office0/',
'/data/coding/eval_replica/Replica/office1/',
'/data/coding/eval_replica/Replica/office2/',
'/data/coding/eval_replica/Replica/office3/',
'/data/coding/eval_replica/Replica/office4/',
'/data/coding/eval_replica/Replica/room0/',
'/data/coding/eval_replica/Replica/room1/',
'/data/coding/eval_replica/Replica/room2/',]

In [17]:
import base64
from zhipuai import ZhipuAI
client = ZhipuAI(api_key="")

R_1,R_2,R_3=[],[],[]

In [19]:
R_1

[9]

In [23]:
for i in path_list[5:]:
    jpg_list=list_files_with_extension(i+'results','jpg')

    frame = cv2.imread(jpg_list[0])
    height, width, layers = frame.shape

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')

    # 创建VideoWriter对象，指定输出文件名、编解码器，帧速率和尺寸
    video = cv2.VideoWriter(i+'video.mp4'.format(i), fourcc, 10, (width, height))

    jpg_list=uniform_sample(jpg_list,40)

    for image in jpg_list:
        video.write(cv2.imread(image))

    cv2.destroyAllWindows()
    video.release()

    video_path=i+'/video.mp4'.format(i)
    with open(video_path, 'rb') as video_file:
        video_base = base64.b64encode(video_file.read()).decode('utf-8')

    output_list=[]

    for iidx,j in enumerate(Question_dict[i.split('/')[-2]]):   ### 遍历每个问题

        if iidx<10:
            prompt="Please answer the question:{} based on the video content,please provide the results directly.please answer yes or no,Do not output punctuation marks and use all lowercase characters".format(j[0])
        if iidx>=10 and iidx<20:
            prompt="Please answer the question:{} based on the video content,please provide the results directly.Only provide Arabic numerals".format(j[0])
        if iidx>=20:
            prompt="Please answer the question:{} based on the video content,please provide the results directly.please answer one words,Do not output punctuation marks and use all lowercase characters".format(j[0])



        prompt="Please answer the question:{} based on the video content,Carefully consider each question,please provide the results directly.Do not output any other information,use one words or number".format(j[0])
        response = client.chat.completions.create(
            model="glm-4v-plus-0111",  # 填写需要调用的模型名称
            messages=[
            {
                "role": "user",
                "content": [
                {"type": "video_url","video_url": {"url" : video_base}},
                {"type": "text","text": prompt}]}])

        print(response.choices[0].message.content)
        output_list.append(response.choices[0].message.content)

    

    print(output_list)
    output=output_list

    r1,r2,r3=acc_(output,[q[1] for q in Question_dict[i.split('/')[-2]]])

    R_1.append(r1)
    R_2.append(r2)
    R_3.append(r3)



no
yes
yes
yes
yes
yes
yes
no
yes
no
1
2
seven
3
2
2
0
2
2
2
cabinet
chair
side table
tv
table
table
vase
tv
tv
side table
['no', 'yes', 'yes', 'yes', 'yes', 'yes', 'yes', 'no', 'yes', 'no', '1', '2', 'seven', '3', '2', '2', '0', '2', '2', '2', 'cabinet', 'chair', 'side table', 'tv', 'table', 'table', 'vase', 'tv', 'tv', 'side table']


ValueError: invalid literal for int() with base 10: 'seven'

In [8]:
i.split('/')[-2]

'office0'

In [26]:
R_2

[2, 2, 3, 3, 2]

In [25]:
print(sum(R_1)/50)
print(sum(R_2)/50)
print(sum(R_3)/50)

print(sum(R_1+R_2+R_3)/150)

0.72
0.24
0.0
0.32


In [5]:
Question_dict={
    'office0':
    [
         ["Is there a trash can in this scene?",'yes'],
         ["Is there a door in this scene?",'yes'],
         ["Is there a chair in this scene?",'yes'],
         ["Is there a sofa in this scene?",'yes'],
         ["Is there a table in this scene?",'yes'],
         ["Is there a cup in this scene?",'no'],
         ["Is there a carpet in this scene?",'yes'],
         ["Is there a phone in this scene?",'yes'],
         ["Is there a pen in this scene?",'no'],
         ["Is there a bag in this scene?",'yes'],

         ['How many trash can are there in this scene?', 2],
         ['How many door are there in this scene?', 1],
         ['How many chair are there in this scene?', 2],
         ['How many table are there in this scene?', 1],
         ['How many sofa are there in this scene?', 4],
         ['How many carpet are there in this scene?', 1],
         ['How many watch are there in this scene?', 1],
         ['How many toy are there in this scene?', 1],
         ['How many bag are there in this scene?', 1],
         ['How many blackboard are there in this scene?', 1],

         ['What is the closest object from the door?', 'trash can'],
         ['What is the farthest object from the trash can?', 'blackboard'],
         ['What is the closest object from the chair?', 'chair'],
         ['What is the farthest object from the door?', 'blackboard'],
         ['What is the closest object from the trash can?', 'trash can'],
         ['What is the closest object from the sofa?', 'sofa'],
         ['What is the closest object from the table?', 'bag'],
         ['What is the farthest object from the blackboard?', 'door'],
         ['What is the farthest object from the sofa?', 'watch'],
         ['What is the closest object from the bag?', 'table']],
    
    'office1':
    [
         ["Is there a trash can in this scene?",'yes'],
         ["Is there a door in this scene?",'yes'],
         ["Is there a chair in this scene?",'no'],
         ["Is there a sofa in this scene?",'no'],
         ["Is there a book in this scene?",'yes'],
         ["Is there a pillow in this scene?",'yes'],
         ["Is there a screen in this scene?",'yes'],
         ["Is there a towel in this scene?",'yes'],
         ["Is there a pen in this scene?",'yes'],
         ["Is there a bag in this scene?",'no'],

         ['How many trash can are there in this scene?', 2],
         ['How many door are there in this scene?', 1],
         ['How many pillow are there in this scene?', 4],
         ['How many screen are there in this scene?', 2],
         ['How many watch are there in this scene?', 1],
         ['How many towel are there in this scene?', 1],
         ['How many watch are there in this scene?', 1],
         ['How many toy are there in this scene?', 0],
         ['How many bag are there in this scene?', 0],
         ['How many blackboard are there in this scene?', 1],

         ['What is the closest object from the door?', 'trash can'],
         ['What is the farthest object from the trash can?', 'pillow'],
         ['What is the closest object from the pillow?', 'pillow'],
         ['What is the farthest object from the door?', 'pillow'],
         ['What is the closest object from the trash can?', 'trash can'],
         ['What is the closest object from the screen?', 'table'],
         ['What is the closest object from the table?', 'screen'],
         ['What is the farthest object from the blackboard?', 'screen'],
         ['What is the farthest object from the watch?', 'screen'],
         ['What is the closest object from the pen?', 'blackboard']],

    'office2':
    [
         ["Is there a trash can in this scene?",'yes'],
         ["Is there a door in this scene?",'yes'],
         ["Is there a chair in this scene?",'yes'],
         ["Is there a sofa in this scene?",'yes'],
         ["Is there a book in this scene?",'no'],
         ["Is there a pillow in this scene?",'yes'],
         ["Is there a screen in this scene?",'yes'],
         ["Is there a towel in this scene?",'no'],
         ["Is there a pen in this scene?",'no'],
         ["Is there a bag in this scene?",'no'],

         ['How many trash can are there in this scene?', 2],
         ['How many door are there in this scene?', 1],
         ['How many pillow are there in this scene?', 5],
         ['How many screen are there in this scene?', 1],
         ['How many chair are there in this scene?', 5],
         ['How many table are there in this scene?', 3],
         ['How many watch are there in this scene?', 0],
         ['How many toy are there in this scene?', 0],
         ['How many bag are there in this scene?', 0],
         ['How many blackboard are there in this scene?', 0],

         ['What is the closest object from the door?', 'chair'],
         ['What is the farthest object from the trash can?', 'sofa'],
         ['What is the closest object from the pillow?', 'sofa'],
         ['What is the farthest object from the door?', 'pillow'],
         ['What is the closest object from the trash can?', 'trash can'],
         ['What is the closest object from the chair?', 'table'],
         ['What is the closest object from the table?', 'chair'],
         ['What is the farthest object from the screen?', 'door'],
         ['What is the farthest object from the sofa?', 'chair'],
         ['What is the closest object from the sofa?', 'pillow']],

    
    'office3':
    [
         ["Is there a trash can in this scene?",'yes'],
         ["Is there a door in this scene?",'yes'],
         ["Is there a chair in this scene?",'yes'],
         ["Is there a sofa in this scene?",'yes'],
         ["Is there a book in this scene?",'no'],
         ["Is there a pillow in this scene?",'yes'],
         ["Is there a screen in this scene?",'yes'],
         ["Is there a towel in this scene?",'no'],
         ["Is there a watch in this scene?",'yes'],
         ["Is there a bag in this scene?",'no'],

         ['How many trash can are there in this scene?', 2],
         ['How many door are there in this scene?', 1],
         ['How many pillow are there in this scene?', 4],
         ['How many screen are there in this scene?', 1],
         ['How many chair are there in this scene?', 9],
         ['How many table are there in this scene?', 1],
         ['How many watch are there in this scene?', 1],
         ['How many sofa are there in this scene?', 2],
         ['How many bag are there in this scene?', 0],
         ['How many blackboard are there in this scene?', 0],

         ['What is the closest object from the door?', 'chair'],
         ['What is the farthest object from the trash can?', 'watch'],
         ['What is the closest object from the pillow?', 'sofa'],
         ['What is the farthest object from the door?', 'watch'],
         ['What is the closest object from the trash can?', 'trash can'],
         ['What is the closest object from the chair?', 'table'],
         ['What is the closest object from the table?', 'chair'],
         ['What is the farthest object from the screen?', 'trash can'],
         ['What is the farthest object from the sofa?', 'watch'],
         ['What is the closest object from the sofa?', 'pillow']],

    
    'office4':
    [
         ["Is there a trash can in this scene?",'yes'],
         ["Is there a door in this scene?",'yes'],
         ["Is there a chair in this scene?",'yes'],
         ["Is there a sofa in this scene?",'no'],
         ["Is there a book in this scene?",'no'],
         ["Is there a pillow in this scene?",'no'],
         ["Is there a screen in this scene?",'yes'],
         ["Is there a towel in this scene?",'no'],
         ["Is there a watch in this scene?",'yes'],
         ["Is there a bag in this scene?",'no'],

         ['How many trash can are there in this scene?', 2],
         ['How many door are there in this scene?', 1],
         ['How many pillow are there in this scene?', 0],
         ['How many screen are there in this scene?', 1],
         ['How many chair are there in this scene?', 12],
         ['How many table are there in this scene?', 1],
         ['How many watch are there in this scene?', 1],
         ['How many sofa are there in this scene?', 0],
         ['How many bag are there in this scene?', 0],
         ['How many blackboard are there in this scene?', 0],

         ['What is the closest object from the door?', 'trash can'],
         ['What is the farthest object from the trash can?', 'watch'],
         ['What is the closest object from the chair?', 'table'],
         ['What is the farthest object from the door?', 'chair'],
         ['What is the closest object from the trash can?', 'trash can'],
         ['What is the closest object from the chair?', 'table'],
         ['What is the closest object from the table?', 'chair'],
         ['What is the farthest object from the screen?', 'door'],
         ['What is the farthest object from the watch?', 'door'],
         ['What is the closest object from the screen?', 'table']],

    'room0':
    [
         ["Is there a trash can in this scene?",'no'],
         ["Is there a door in this scene?",'yes'],
         ["Is there a chair in this scene?",'yes'],
         ["Is there a sofa in this scene?",'yes'],
         ["Is there a book in this scene?",'no'],
         ["Is there a pillow in this scene?",'yes'],
         ["Is there a window in this scene?",'yes'],
         ["Is there a towel in this scene?",'no'],
         ["Is there a lamp in this scene?",'yes'],
         ["Is there a bag in this scene?",'no'],

         ['How many trash can are there in this scene?', 0],    
         ['How many door are there in this scene?', 1],
         ['How many pillow are there in this scene?', 8],
         ['How many window are there in this scene?', 3],
         ['How many chair are there in this scene?', 2],
         ['How many table are there in this scene?', 1],
         ['How many watch are there in this scene?', 0],
         ['How many sofa are there in this scene?', 4],
         ['How many lamp are there in this scene?', 2],
         ['How many blackboard are there in this scene?', 0],

         ['What is the closest object from the door?', 'cabinet'],
         ['What is the farthest object from the cabinet?', 'window'],
         ['What is the closest object from the chair?', 'chair'],
         ['What is the farthest object from the door?', 'lamp'],
         ['What is the closest object from the sofa?', 'pillow'],
         ['What is the closest object from the pillow?', 'sofa'],
         ['What is the closest object from the table?', 'chair'],
         ['What is the farthest object from the window?', 'door'],
         ['What is the farthest object from the lamp?', 'door'],
         ['What is the closest object from the lamp?', 'sofa']],

    
    'room1':
    [
         ["Is there a trash can in this scene?",'no'],
         ["Is there a door in this scene?",'yes'],
         ["Is there a chair in this scene?",'no'],
         ["Is there a bed in this scene?",'yes'],
         ["Is there a book in this scene?",'no'],
         ["Is there a pillow in this scene?",'yes'],
         ["Is there a window in this scene?",'yes'],
         ["Is there a towel in this scene?",'no'],
         ["Is there a lamp in this scene?",'yes'],
         ["Is there a bag in this scene?",'no'],

         ['How many trash can are there in this scene?', 0],    
         ['How many door are there in this scene?', 1],
         ['How many pillow are there in this scene?', 5],
         ['How many window are there in this scene?', 2],
         ['How many chair are there in this scene?', 0],
         ['How many bed are there in this scene?', 1],
         ['How many picture are there in this scene?', 1],
         ['How many cabinet are there in this scene?', 3],
         ['How many lamp are there in this scene?', 2],
         ['How many blackboard are there in this scene?', 0],

         ['What is the closest object from the door?', 'cabinet'],
         ['What is the farthest object from the cabinet?', 'window'],
         ['What is the closest object from the bed?', 'pillow'],
         ['What is the farthest object from the door?', 'window'],
         ['What is the closest object from the cabinet?', 'lamp'],
         ['What is the closest object from the pillow?', 'bed'],
         ['What is the closest object from the picture?', 'bed'],
         ['What is the farthest object from the window?', 'door'],
         ['What is the farthest object from the lamp?', 'door'],
         ['What is the closest object from the lamp?', 'cabinet']],
    

    'room2':
    [
         ["Is there a trash can in this scene?",'no'],
         ["Is there a door in this scene?",'yes'],
         ["Is there a chair in this scene?",'yes'],
         ["Is there a bed in this scene?",'no'],
         ["Is there a book in this scene?",'no'],
         ["Is there a table in this scene?",'yes'],
         ["Is there a window in this scene?",'yes'],
         ["Is there a towel in this scene?",'no'],
         ["Is there a lamp in this scene?",'no'],
         ["Is there a shelf in this scene?",'yes'],

         ['How many trash can are there in this scene?', 0],    
         ['How many door are there in this scene?', 1],
         ['How many shelf are there in this scene?', 1],
         ['How many window are there in this scene?', 2],
         ['How many chair are there in this scene?', 8],
         ['How many bed are there in this scene?', 0],
         ['How many picture are there in this scene?', 1],
         ['How many cabinet are there in this scene?', 0],
         ['How many lamp are there in this scene?', 0],
         ['How many vase are there in this scene?', 1],

         ['What is the closest object from the door?', 'shelf'],
         ['What is the farthest object from the shelf?', 'picture'],
         ['What is the closest object from the table?', 'chair'],
         ['What is the farthest object from the door?', 'window'],
         ['What is the closest object from the shelf?', 'vase'],
         ['What is the closest object from the chair?', 'table'],
         ['What is the closest object from the picture?', 'chair'],
         ['What is the farthest object from the window?', 'door'],
         ['What is the farthest object from the picture?', 'door'],
         ['What is the farthest object from the table?', 'door']],
       
}

In [11]:
r1,r2,r3=acc_(output,[q[1] for q in Question_dict['office0']])

In [12]:
r1

8

In [13]:
r2

4

In [14]:
r3

0